# Deliverable 3 – Battle Simulator

INF201, NMBU

Amalie Berntsen and Sander Vikse

GitHub: https://github.com/amaliebern-code/inf201--deliverable-3

We have built upon the `Character` class from Lecture 8 to create a battle simulator. The code is organized into four files: `character.py`, `dice.py`, `items.py`, and `battle.py`. Below, we present the functionality of each class and simulate several battles.

In [80]:
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon
from battle import Battle
from typing import Optional
import random

## 1. Character

A `Character` has a name, HP, maximum HP, base damage, and an optional weapon. We added `take_damage()`, `heal()`, and `is_alive`, and updated `attack()` so that it calls the target's `take_damage()` method rather than directly modifying `_hp`.

In [81]:
# Object-oriented game simulator
# Turid Torheim, NMBU, 2026

class Character:
    """Represents a character in the RPG"""

    def __init__(
        self,
        name: str,
        hp_max: int = 100,
        damage: int = 10,
        weapon: Optional[Weapon] = None,
    ) -> None:
        """
        Initialise the Character

        Args:
            name (str): The name of the character.
            hp_max (int): Max hit points (default = 100)
            damage (int): Damage delt when character attacks (default = 10)
        """
        self.name = name
        self._hp = hp_max  # Note: _hp indicated private attribute
        self.hp_max = hp_max
        self.damage = damage
        self.weapon = weapon

    def take_damage(self, amount: int) -> None:
        """Reduces character's HP by amount, but not below 0"""
        self._hp = max(self._hp - amount, 0)  # HP cannot go below 0

    def heal(self, amount: int) -> None:
        """Increase character's HP by amount, but not above hp_max"""
        self._hp = min(self._hp + amount, self.hp_max)

    def attack(self, target) -> None:
        """Attack and deal damage to target character"""
        if self.weapon is not None:
            damage = self.weapon.roll_damage()
        else:
            damage = self.damage
        target.take_damage(damage)

    def introduce(self) -> None:
        """Creates string that introduces character"""
        print(f"Hi, my name is {self.name}!")

    def __repr__(self) -> str:
        """Representation of character object"""
        return f"<Character name={self.name} hp={self.hp} damage={self.damage}>"

    def __str__(self) -> str:
        """String representation of character object"""
        return f"Character: {self.name} with {self.hp} HP and {self.damage} damage"

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe edits."""
        return self._hp
    
    @property
    def is_alive(self) -> bool:
        """Read-only property: True if character is alive (HP > 0), False otherwise."""
        return self._hp > 0

class Warrior(Character):
    """A defensive character with armor that reduves incoming damage"""

    def __init__(
            self,
            name: str,
            hp_max: int= 100,
            damage: int = 10, 
            weapon: Optional [Weapon] = None,
            armor: int = 5, 
    ) -> None:
        """
        Initialise the Warrior

        Args:
            name (str): The name of the character.
            hp_max (int): Max hit points (default = 100)
            damage (int): Damage delt when character attacks (default = 10)
            armor (int): Reduces incoming damage by this amount (default = 5)
        """
        super().__init__(name, hp_max, damage, weapon)
        self.armor = armor

    def take_damage(self, amount: int) -> None:
        """Reduce the incoming damage by armor before applying it"""
        reduced = max(amount- self.armor, 0)  # Armor cant turn damage into healing
        super().take_damage(reduced)

class Mage(Character):
    """A character that can cast spells to deal damage"""

    def __init__(
            self,
            name: str,
            hp_max: int= 100,
            damage: int = 10, 
            weapon: Optional [Weapon] = None,
            mana: int = 30,
            mana_max: int=30, 
    ) ->None:
        """
        Initialise the Mage

        Args:
            mana (int): Mana the mage starts with (default = 30)
            mana_max (int): Max mana (default = 30)
            The other arguments are the same as for Character.
        """
        super().__init__(name, hp_max, damage, weapon)
        self.mana = mana
        self.mana_max = mana_max

    def attack(self, target) -> None:
        """Cast a spell if there is enough mana, otherwise use the base attack"""
        if self.mana >= 10:
            self.mana -= 10
            target.take_damage(self.damage * 2)  # Spell: double damage, no weapon
        else:
            super().attack(target)  # Base attack, uses weapon if equipped
            self.mana = min(self.mana + 5, self.mana_max)

if __name__ == "__main__":
    warrior = Warrior("Warrior")
    mage = Mage("Mage")

    mage.attack(warrior)     # spell: 20 damage, armor blocks 5
    print(warrior.hp)        # 85
    print(mage.mana)         # 20
    mage.attack(warrior)
    mage.attack(warrior)
    print(mage.mana)         # 0
    mage.attack(warrior)     # no mana: base attack of 10, armor blocks 5
    print(warrior.hp)        # 50
    print(mage.mana)         # 5

85
20
0
50
5


* **Comment:** Here, we extended the core functionality of `Character`. By using `@property` for `hp` and `is_alive`, we ensure encapsulation so that the state cannot be unsafely overwritten from the outside. Damage and healing are handled in a controlled manner using `take_damage()` and `heal()`.

## 2. Dice

A die with a customizable number of sides (default is 6). It can be rolled using the `roll()` method or by calling the object directly as a function.

In [82]:
class Dice: 
    """Represents a dice with a given number of sides"""

    def __init__(self,sides : int = 6) -> None: 
        """Initialise the Dice
    
        Args: 
            sides (int): Number of sides, must be at least 2 (default = 6)
        """
        if sides < 2: 
            raise ValueError("Dice must have at least 2 sides")
        self.sides = sides
    
    def roll(self) -> int: 
        """Roll the dice and return a random number between 1 and sides"""
        return random.randint(1, self.sides)
    
    def __call__(self) -> int:
        """Allow the dice to be called like a function to roll it"""
        return self.roll()

    def __repr__(self) -> str: 
        """Representation of the Dice object"""
        return f"Dice(sides={self.sides})"
    
if __name__ == "__main__":
    d6 = Dice()  # Default 6-sided dice
    d20 = Dice(sides=20)  # 20-sided dice

    print(d6)           #Dice sides=6
    print(d20)          #Dice sides=20
    print(d6.roll())  # Random number between 1 and 6
    print(d20.roll())  # Random number between 1 and 20

Dice(sides=6)
Dice(sides=20)
6
3


* **Comment:** The `Dice` class manages randomness in the battle system. By implementing `__call__`, the dice object can be invoked directly like a function (`d6()`), and `__repr__` provides a clean technical string representation during debugging.

## 3. Weapon

A weapon has a name and a damage die (`damage_dice`). The die determines the damage dealt by the weapon.

In [83]:
class Weapon:
    """Represents a weapon that a character can use to attack"""

    def __init__(self, name: str, damage_dice: Dice) -> None:
        """
        Initialise the Weapon

        Args:
            name (str): The name of the weapon.
            damage_dice (Dice): The dice rolled to decide the damage.
        """
        self.name = name
        self.damage_dice = damage_dice

    def roll_damage(self) -> int:
        """Roll the weapon's dice and return the damage"""
        return self.damage_dice.roll()

    def __repr__(self) -> str:
        """Representation of weapon object"""
        return f"Weapon(name={self.name}, damage_dice={self.damage_dice})"


if __name__ == "__main__":
    sword = Weapon("Sword", Dice(sides=8))
    print(sword)                # Weapon(name=Sword, damage_dice=Dice(sides=8))
    print(sword.roll_damage())  # random number from 1 to 8

Weapon(name=Sword, damage_dice=Dice(sides=8))
2


* **Comment:** `Weapon` demonstrates composition: a weapon *has a* die (`damage_dice`). When `roll_damage()` is called, responsibility is delegated to the die's own `roll()` method.

## 4. Warrior and Mage

Both classes inherit from `Character`.

`Warrior` has armor (`armor`) that reduces incoming damage:

In [68]:
warrior = Warrior("Warrior")
print("Rustning:", warrior.armor)

warrior.take_damage(20)
print(warrior.hp)

Rustning: 5
85


`Mage` has mana. If the mage has at least 10 mana, it casts a spell dealing double damage at the cost of 10 mana. Otherwise, it performs a standard attack and restores 5 mana.

In [69]:
mage = Mage("Mage")
target = Character("Target")

for i in range(6):
    mage.attack(target)
    print("HP hos target:", target.hp, " Mana:", mage.mana)

HP hos target: 80  Mana: 20
HP hos target: 60  Mana: 10
HP hos target: 40  Mana: 0
HP hos target: 30  Mana: 5
HP hos target: 20  Mana: 10
HP hos target: 0  Mana: 0


* **Comment:** Here, inheritance and method overriding (polymorphism) are used. `Warrior` reuses the parent class logic using `super()`, but reduces incoming damage with `armor`. `Mage` is controlled by its own resource management (`mana`), casting powerful spells until mana is depleted before recharging with standard attacks.

The first three attacks deal 20 damage each. Once mana is depleted, the mage reverts to standard attacks (dealing 10 damage) until enough mana is restored for another spell.

## 5. Battle

`Battle` manages turn-based combat between two characters. `play_round()` executes one round of combat, and `simulate_battle()` runs until one character is defeated.

In [75]:
class Battle: 
    """Manages a turn-based battle between two characters"""

    def __init__(self, hero: Character, enemy: Character) -> None: 
        """Initialise the Battle

        Args: 
            hero (Character): The hero character
            enemy (Character): The enemy character
        """
        self.hero = hero
        self.enemy = enemy
        self.round_number = 0

    def play_round(self) -> None:
        """Play one round: hero attacks, and enemy attacks back if still alive"""
        self.round_number += 1
        self.hero.attack(self.enemy)
        if self.enemy.is_alive:
            self.enemy.attack(self.hero)

        print(
            f"Round {self.round_number}: "
            f"{self.hero.name} {self.hero.hp} HP, "
            f"{self.enemy.name} {self.enemy.hp} HP"
        )

    def simulate_battle(self, max_rounds: int = 100) -> Optional[Character]:
        """
        
        Play rounds until one character is no longer alive
        
        Args: 
            max_round (int) : Stop after this many rounds to avoid and endless

        Returns:
            The winning character, or None if nobody won within max_rounds. 
        """

        while self.hero.is_alive and self.enemy.is_alive:
            if self.round_number >= max_rounds: 
                print(f"No winner after {max_rounds} rounds")
                return None
            self.play_round()

        winner = self.hero if self.hero.is_alive else self.enemy
        print(f"{winner.name} wins after {self.round_number} rounds!")
        return winner

if __name__ == "__main__":
    from character import Mage, Warrior
    from dice import Dice
    from items import Weapon

    sword = Weapon("Sword", Dice(sides=12))
    hero = Warrior("Aragorn", weapon=sword)
    enemy = Mage("Saruman")

    battle = Battle(hero, enemy)
    battle.simulate_battle()


Round 1: Aragorn 85 HP, Saruman 93 HP
Round 2: Aragorn 70 HP, Saruman 86 HP
Round 3: Aragorn 55 HP, Saruman 81 HP
Round 4: Aragorn 50 HP, Saruman 80 HP
Round 5: Aragorn 45 HP, Saruman 73 HP
Round 6: Aragorn 30 HP, Saruman 65 HP
Round 7: Aragorn 25 HP, Saruman 63 HP
Round 8: Aragorn 20 HP, Saruman 59 HP
Round 9: Aragorn 5 HP, Saruman 51 HP
Round 10: Aragorn 0 HP, Saruman 44 HP
Saruman wins after 10 rounds!


* **Comment:** The `Battle` class orchestrates the entire game system by connecting two characters[cite: 8]. `play_round()` handles the turn-based attack order and terminates immediately if the enemy dies on the first hit[cite: 8]. Additionally, `simulate_battle()` includes a `max_rounds` safeguard to prevent infinite loops if neither character can defeat the other[cite: 8].

## 6. Battles

### Without Weapons

In [76]:
battle = Battle(Character("Hero"), Character("Enemy"))
winner = battle.simulate_battle()

Round 1: Hero 90 HP, Enemy 90 HP
Round 2: Hero 80 HP, Enemy 80 HP
Round 3: Hero 70 HP, Enemy 70 HP
Round 4: Hero 60 HP, Enemy 60 HP
Round 5: Hero 50 HP, Enemy 50 HP
Round 6: Hero 40 HP, Enemy 40 HP
Round 7: Hero 30 HP, Enemy 30 HP
Round 8: Hero 20 HP, Enemy 20 HP
Round 9: Hero 10 HP, Enemy 10 HP
Round 10: Hero 10 HP, Enemy 0 HP
Hero wins after 10 rounds!


Both characters deal 10 damage per round. The hero attacks first and therefore always wins in round 10.

### With and Without Weapons

Here, the enemy is equipped with a sword using a 12-sided die (d12), while the hero attacks unarmed.

In [77]:
hero = Character("Hero")
enemy = Character("Enemy", weapon=Weapon("Sword", Dice(sides=12)))

battle = Battle(hero, enemy)
winner = battle.simulate_battle()

Round 1: Hero 89 HP, Enemy 90 HP
Round 2: Hero 77 HP, Enemy 80 HP
Round 3: Hero 71 HP, Enemy 70 HP
Round 4: Hero 61 HP, Enemy 60 HP
Round 5: Hero 49 HP, Enemy 50 HP
Round 6: Hero 47 HP, Enemy 40 HP
Round 7: Hero 45 HP, Enemy 30 HP
Round 8: Hero 43 HP, Enemy 20 HP
Round 9: Hero 37 HP, Enemy 10 HP
Round 10: Hero 37 HP, Enemy 0 HP
Hero wins after 10 rounds!


The hero wins even without a weapon. The sword deals between 1 and 12 damage (averaging 6.5), which is less than the hero's fixed 10 base damage. Equipping a weapon does not automatically make a character stronger.

### Warrior vs. Mage (Without Weapons)

In [78]:
battle = Battle(Warrior("Warrior"), Mage("Mage"))
winner = battle.simulate_battle()

Round 1: Warrior 85 HP, Mage 90 HP
Round 2: Warrior 70 HP, Mage 80 HP
Round 3: Warrior 55 HP, Mage 70 HP
Round 4: Warrior 50 HP, Mage 60 HP
Round 5: Warrior 45 HP, Mage 50 HP
Round 6: Warrior 30 HP, Mage 40 HP
Round 7: Warrior 25 HP, Mage 30 HP
Round 8: Warrior 20 HP, Mage 20 HP
Round 9: Warrior 5 HP, Mage 10 HP
Round 10: Warrior 5 HP, Mage 0 HP
Warrior wins after 10 rounds!


The warrior loses 15 HP in each of the first three rounds because the mage casts spells dealing 20 damage (reduced by 5 due to armor). After that, the mage is low on mana and deals only 5 effective damage per turn. The warrior wins with 5 HP remaining, and since neither has a weapon, the outcome is deterministic.

### Warrior vs. Mage (With Weapons)

In [79]:
warrior = Warrior("Warrior", weapon=Weapon("Sword", Dice(sides=12)))
mage = Mage("Mage", weapon=Weapon("Staff", Dice(sides=6)))

battle = Battle(warrior, mage)
winner = battle.simulate_battle()

Round 1: Warrior 85 HP, Mage 93 HP
Round 2: Warrior 70 HP, Mage 92 HP
Round 3: Warrior 55 HP, Mage 87 HP
Round 4: Warrior 55 HP, Mage 85 HP
Round 5: Warrior 54 HP, Mage 75 HP
Round 6: Warrior 39 HP, Mage 73 HP
Round 7: Warrior 39 HP, Mage 61 HP
Round 8: Warrior 39 HP, Mage 58 HP
Round 9: Warrior 24 HP, Mage 46 HP
Round 10: Warrior 24 HP, Mage 43 HP
Round 11: Warrior 24 HP, Mage 42 HP
Round 12: Warrior 9 HP, Mage 38 HP
Round 13: Warrior 9 HP, Mage 28 HP
Round 14: Warrior 9 HP, Mage 25 HP
Round 15: Warrior 0 HP, Mage 22 HP
Mage wins after 15 rounds!


Now both characters roll dice for damage, introducing variance to the outcome. Running the cell multiple times results in victories for both the warrior and the mage. The battle lasts longer than without weapons because the sword's average damage is lower than 10. The mage's staff has minimal impact since spells do not use weapons and the warrior's armor blocks most physical staff damage.

---
---
---
## AI Interaction Log & Declaration

Throughout this project, AI was utilized as an interactive learning mentor and thought partner to clarify object-oriented principles, architecture design, and syntax without generating full direct code implementations.

### Summary of AI Discussions by Task:

* **Task 1 & 2: Encapsulation & Magic Methods**[cite: 5]
  * **Question:** *What is printed when calling `print(Dice())` without `__repr__`, and what should it output?*[cite: 3]
  * **Guidance:** Clarified that default Python behavior outputs memory addresses (e.g., `<__main__.Dice object... >`), and implemented `__repr__` to return formatted strings like `Dice(sides=6)`[cite: 3, 5].
  * **Encapsulation & `@property`:** Discussed read-only properties (`hp`, `is_alive`) to protect private attributes (`_hp`) while providing dynamic evaluation[cite: 3, 5].

* **Task 3: Composition & Modules**[cite: 6]
  * **Question:** *What owns what between `Character`, `Weapon`, and `Dice`?*
  * **Guidance:** Established object composition: `Character` holds an optional `Weapon`, while `Weapon` holds a `Dice` instance[cite: 3, 6].
  * **Imports & Optional Arguments:** Clarified cross-file module imports and setting `weapon = None` as a default optional parameter[cite: 3, 6].

* **Task 4: Inheritance & Specialization**[cite: 6]
  * **Question:** *How do subclasses inherit and reuse initialization logic from the parent class?*
  * **Guidance:** Reviewed `class Warrior(Character):` syntax and `super().__init__(...)`[cite: 4, 6].
  * **Method Overriding:** Discussed mitigating damage with `max(0, damage - armor)` for `Warrior`, and conditional resource logic (`mana`) for `Mage`[cite: 4, 6].

* **Task 5: Battle Simulation & Edge Cases**[cite: 6]
  * **Question:** *How do we determine turn order, death conditions, and prevent infinite loops?*
  * **Guidance:** Structured `play_round()` to check `enemy.is_alive` before counter-attacking[cite: 3, 6]. Added a `max_rounds` safeguard to `simulate_battle()` to handle endless draws[cite: 3, 6].

### Learning Outcome:
Using AI as a thought partner helped reinforce object-oriented programming concepts, clean project structure, and safe state management in Python.
---
